# 02 — Magnetic Susceptibility Inversion (SimPEG)

This notebook builds the inversion mesh and SimPEG survey/simulation objects, then runs:
1. A smooth (L2) inversion to recover a stable susceptibility model
2. A sparse (IRLS) inversion to sharpen geological boundaries

Outputs from this notebook are saved to `data/processed/` and used directly by `03_prospectivity.ipynb` for target mapping.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

from simpeg import data as simpeg_data

# Resolve repository paths relative to notebook folder.
REPO_ROOT = Path.cwd().resolve().parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from inversion_setup import build_mesh, build_survey, build_simulation
from run_inversion import run_smooth_inversion, run_sparse_inversion

In [ ]:
# Load processed observation file created in notebook 01.
obs_path = REPO_ROOT / "data" / "processed" / "red_lake_tmi_obs.npz"
if not obs_path.exists():
    raise FileNotFoundError(
        f"Observation file not found: {obs_path}. Run 01_data_prep.ipynb first."
    )

obs = np.load(obs_path)
receiver_locs = obs["receiver_locations"]
dobs = obs["data"]
std = obs["uncertainties"]

x_obs = np.unique(receiver_locs[:, 0])
y_obs = np.unique(receiver_locs[:, 1])

print(f"Loaded observations: {dobs.size:,} points")
print(f"X range: {x_obs.min():.2f} to {x_obs.max():.2f} m")
print(f"Y range: {y_obs.min():.2f} to {y_obs.max():.2f} m")

In [ ]:
# 1) Build the inversion mesh and print dimensions.
mesh = build_mesh(x_obs, y_obs, core_cell_size=100.0, padding_cells=8, padding_factor=1.5)

print("Mesh shape (nCx, nCy, nCz):", mesh.shape_cells)
print("Total cells:", mesh.n_cells)
print("Cell widths hx, hy, hz (first 5):")
print(mesh.h[0][:5], mesh.h[1][:5], mesh.h[2][:5])

In [ ]:
# 2) Plot a 2D slice of the empty mesh to check geometry.
# We display an XY view at the top layer by plotting cell-center locations.
cc = mesh.cell_centers
z_top = cc[:, 2].max()
xy_top = cc[np.isclose(cc[:, 2], z_top)]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(xy_top[:, 0], xy_top[:, 1], s=1, c="k", alpha=0.4)
ax.set_title("Mesh QC: Top-Layer Cell Centers (XY)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")
plt.show()

In [ ]:
# 3) Build survey + simulation.
survey = build_survey(obs_path)

# For now we use all cells as active (no topography file yet).
actind = np.ones(mesh.n_cells, dtype=bool)

simulation = build_simulation(mesh, survey, actind)

# Build SimPEG Data object (observed data + standard deviations).
data_obj = simpeg_data.Data(survey=survey, dobs=dobs, standard_deviation=std)
print("Survey/simulation/data objects built successfully.")

In [ ]:
# 4) Run smooth inversion.
m_smooth = run_smooth_inversion(
    simulation=simulation,
    data_obj=data_obj,
    mesh=mesh,
    actind=actind,
    starting_susceptibility=1e-4,
)
print("Smooth inversion completed.")
print("Saved: data/processed/susceptibility_smooth.npy")

In [ ]:
# 5) Plot smooth inversion misfit curve (phi_d and phi_m vs iteration).
# SimPEG directive outputs vary by version, so this cell tries common file names.
processed_dir = REPO_ROOT / "data" / "processed"
misfit_candidates = sorted(processed_dir.glob("**/*iteration*")) + sorted(processed_dir.glob("**/*output*"))

phi_d = []
phi_m = []

for f in misfit_candidates:
    if f.suffix.lower() in {".txt", ".csv", ".json", ".npz"}:
        try:
            if f.suffix.lower() == ".npz":
                d = np.load(f)
                if "phi_d" in d and "phi_m" in d:
                    phi_d = list(np.asarray(d["phi_d"]).ravel())
                    phi_m = list(np.asarray(d["phi_m"]).ravel())
                    break
            elif f.suffix.lower() in {".txt", ".csv"}:
                text = f.read_text(errors="ignore")
                lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
                for ln in lines:
                    low = ln.lower()
                    if "phi_d" in low and "phi_m" in low:
                        parts = ln.replace(",", " ").split()
                        vals = [p for p in parts if p.replace(".", "", 1).replace("-", "", 1).isdigit()]
                        if len(vals) >= 2:
                            phi_d.append(float(vals[-2]))
                            phi_m.append(float(vals[-1]))
        except Exception:
            pass

fig, ax = plt.subplots(figsize=(8, 5))
if phi_d and phi_m:
    it = np.arange(1, len(phi_d) + 1)
    ax.plot(it, phi_d, "o-", label="phi_d (data misfit)")
    ax.plot(it, phi_m, "s-", label="phi_m (model objective)")
else:
    # Fallback placeholder when no iteration logs are available.
    ax.text(0.5, 0.5, "No iteration log found for phi_d/phi_m.", ha="center", va="center")
    ax.set_xticks([])
    ax.set_yticks([])
ax.set_title("Smooth Inversion Misfit Curve")
ax.set_xlabel("Iteration")
ax.set_ylabel("Objective value")
ax.legend(loc="best") if phi_d and phi_m else None
plt.show()

In [ ]:
# Helper: convert active-cell model vector -> full 3D array for plotting.
def model_to_3d(model_active, mesh_obj, active_mask):
    model_full = np.zeros(mesh_obj.n_cells, dtype=float)
    model_full[active_mask] = model_active
    return model_full.reshape(mesh_obj.shape_cells, order="F")


def depth_slice(model_3d, mesh_obj, depth_m=200.0):
    # Find z-index closest to requested depth below surface (z=0 assumed near top).
    z_cc = mesh_obj.cell_centers_z
    target_z = -abs(depth_m)
    kz = int(np.argmin(np.abs(z_cc - target_z)))
    return model_3d[:, :, kz], z_cc[kz]

smooth_3d = model_to_3d(m_smooth, mesh, actind)
smooth_slice, smooth_z = depth_slice(smooth_3d, mesh, depth_m=200.0)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(
    smooth_slice.T,
    origin="lower",
    extent=[mesh.cell_centers_x.min(), mesh.cell_centers_x.max(), mesh.cell_centers_y.min(), mesh.cell_centers_y.max()],
    aspect="auto",
    cmap="plasma",
)
cb = plt.colorbar(im, ax=ax)
cb.set_label("Susceptibility (SI)")
ax.set_title(f"Smooth Susceptibility at Depth ~ {abs(smooth_z):.0f} m")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

In [ ]:
# 6) Run sparse inversion starting from smooth model.
m_sparse = run_sparse_inversion(
    simulation=simulation,
    data_obj=data_obj,
    mesh=mesh,
    actind=actind,
    m0=m_smooth,
    p_s=0,
    p_x=1,
    p_y=1,
    p_z=1,
)
print("Sparse inversion completed.")
print("Saved: data/processed/susceptibility_sparse.npy")

In [ ]:
# 7) Plot sparse inversion misfit curve.
misfit_candidates = sorted(processed_dir.glob("**/*iteration*")) + sorted(processed_dir.glob("**/*output*"))

phi_d_sp = []
phi_m_sp = []

for f in misfit_candidates:
    if "sparse" not in f.name.lower() and "irls" not in f.name.lower():
        continue
    if f.suffix.lower() in {".txt", ".csv", ".json", ".npz"}:
        try:
            if f.suffix.lower() == ".npz":
                d = np.load(f)
                if "phi_d" in d and "phi_m" in d:
                    phi_d_sp = list(np.asarray(d["phi_d"]).ravel())
                    phi_m_sp = list(np.asarray(d["phi_m"]).ravel())
                    break
            elif f.suffix.lower() in {".txt", ".csv"}:
                text = f.read_text(errors="ignore")
                lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
                for ln in lines:
                    low = ln.lower()
                    if "phi_d" in low and "phi_m" in low:
                        parts = ln.replace(",", " ").split()
                        vals = [p for p in parts if p.replace(".", "", 1).replace("-", "", 1).isdigit()]
                        if len(vals) >= 2:
                            phi_d_sp.append(float(vals[-2]))
                            phi_m_sp.append(float(vals[-1]))
        except Exception:
            pass

fig, ax = plt.subplots(figsize=(8, 5))
if phi_d_sp and phi_m_sp:
    it = np.arange(1, len(phi_d_sp) + 1)
    ax.plot(it, phi_d_sp, "o-", label="phi_d (data misfit)")
    ax.plot(it, phi_m_sp, "s-", label="phi_m (model objective)")
else:
    ax.text(0.5, 0.5, "No sparse iteration log found for phi_d/phi_m.", ha="center", va="center")
    ax.set_xticks([])
    ax.set_yticks([])
ax.set_title("Sparse Inversion Misfit Curve")
ax.set_xlabel("Iteration")
ax.set_ylabel("Objective value")
ax.legend(loc="best") if phi_d_sp and phi_m_sp else None
plt.show()

In [ ]:
# 8) Side-by-side smooth vs sparse susceptibility slices at ~200 m depth.
sparse_3d = model_to_3d(m_sparse, mesh, actind)
sparse_slice, sparse_z = depth_slice(sparse_3d, mesh, depth_m=200.0)

vmin = min(np.nanmin(smooth_slice), np.nanmin(sparse_slice))
vmax = max(np.nanmax(smooth_slice), np.nanmax(sparse_slice))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

im0 = axes[0].imshow(
    smooth_slice.T,
    origin="lower",
    extent=[mesh.cell_centers_x.min(), mesh.cell_centers_x.max(), mesh.cell_centers_y.min(), mesh.cell_centers_y.max()],
    aspect="auto",
    cmap="plasma",
    vmin=vmin,
    vmax=vmax,
)
axes[0].set_title(f"Smooth Model at ~{abs(smooth_z):.0f} m")
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")

im1 = axes[1].imshow(
    sparse_slice.T,
    origin="lower",
    extent=[mesh.cell_centers_x.min(), mesh.cell_centers_x.max(), mesh.cell_centers_y.min(), mesh.cell_centers_y.max()],
    aspect="auto",
    cmap="plasma",
    vmin=vmin,
    vmax=vmax,
)
axes[1].set_title(f"Sparse Model at ~{abs(sparse_z):.0f} m")
axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Northing (m)")

cb = fig.colorbar(im1, ax=axes, shrink=0.9)
cb.set_label("Susceptibility (SI)")
plt.show()

In [ ]:
# 9) Explicitly save both model vectors to data/processed/.
processed_dir = REPO_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

smooth_path = processed_dir / "susceptibility_smooth.npy"
sparse_path = processed_dir / "susceptibility_sparse.npy"

np.save(smooth_path, m_smooth)
np.save(sparse_path, m_sparse)

print(f"Saved smooth model: {smooth_path}")
print(f"Saved sparse model: {sparse_path}")